### Loading and splitting codes files

In [ ]:
# Loading Markdown files(.md)
from langchain_community.document_loaders import UnstructuredMarkdownLoader

loader = UnstructuredMarkdownLoader("data/knowledge_base.md")
markdown_content = loader.load()
print(markdown_content[0])

In [ ]:
# Loading Python files(.py)
from abc import ABC, abstractmethod
from langchain_community.document_loaders import PythonLoader

class LLM(ABC):
    @abstractmethod
    def complete_sentence(self, prompt: str) -> str:
        pass



# Integrated into RAG applications for writing or fixing code, creating docs, etc.
loader = PythonLoader("chatbot.py")
python_data = loader.load()
print(python_data[0])

In [ ]:
# Splitting code files into chunks
python_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0)

chunks = python_splitter.split_documents(python_data)
for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i+1}:\n{chunk.page_content}\n{'-'*50}")

#### Splitting by language

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, 
    chunk_size=150, 
    chunk_overlap=10)

chunks = python_splitter.split_documents(python_data)
for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i+1}:\n{chunk.page_content}\n{'-'*50}")

In [ ]:
# Create a document loader for rag.py and load it
loader = PythonLoader("rag.py")

python_data = loader.load()
print(python_data[0])

In [ ]:
# Create a Python-aware recursive character splitter
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=300, chunk_overlap=100
)

# Split the Python content into chunks
chunks = python_splitter.split_documents(python_data)

for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i+1}:\n{chunk.page_content}\n")

### Advanced splitting methods

In [ ]:
# Splitting tokens
import tiktoken
from langchain_text_splitters import TokenTextSplitter
example_string = "Mary had a little lamb, its fleece was white as snow."

encoding = tiktoken.encoding_for_model("gpt-4o-mini")
splitter = TokenTextSplitter(encoding=encoding, 
                             chunk_size=10, 
                             chunk_overlap=2)

chunks = splitter.split_text(example_string)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: \n{chunk}\n")

In [ ]:
# Splitting on tokens
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: \nNo tokens: {len(encoding.encode(chunk))}\n{chunk}\n")

In [ ]:
# Semantic splitting
from langchain_penai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker


embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=openai_api_key)

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
   breakpoint_threshold_type = "gradient",
   breakpoint_threshold_amount = 0.8
)


In [ ]:
chunks = semantic_splitter.split_documents(data)
print(chunks[0])

### Exercise:
splitting by token

In [ ]:
# Get the encoding for gpt-4o-mini
encoding = tiktoken.encoding_for_model('gpt-4o-mini')

# Create a token text splitter
token_splitter = TokenTextSplitter(encoding_name=encoding.name, chunk_size=100, chunk_overlap=10)

# Split the PDF into chunks
chunks = token_splitter.split_documents(document)

for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i+1}:\nNo. tokens: {len(encoding.encode(chunk.page_content))}\n{chunk}\n")

- Splitting semantically

In [ ]:
# Instantiate an OpenAI embeddings model
embedding_model = OpenAIEmbeddings(api_key="<OPENAI_API_TOKEN>", model='text-embedding-3-small')

# Create the semantic text splitter with desired parameters
semantic_splitter = SemanticChunker(
    embeddings=embedding_model, breakpoint_threshold_type="gradient", breakpoint_threshold_amount=0.8
)

# Split the document
chunks = semantic_splitter.split_documents(document)
print(chunks[0])

### Optimizing document retrieval

- Sparse retrieval methods
TF-IDF: Encodes documents using the words that make the document unique

- BM25: Helps mitigate high-frequency words from saturating the encoding

In [ ]:
# BM25 retrieval 
from langchain_core.retrievers import BM25Retriever

chunks = [
    "Python was created by Guido van Rossum and first released in 1991.",
    "Python is a popular  language for machine learning (ML).",
    "The PyTorch library is a popular Python library for AI and ML."
]

bm25_retriever = BM25Retriever.from_texts(chunks, k=3)

In [ ]:
results = bm25_retriever.invoke("When was Python created?")
print("Most Relevant Document:")
print(results[0].page_content)

- BM25 in RAG

In [ ]:
retriever = BM25Retriever.from_documents(documents = chunks, 
                                         k=5
                                         )

chain = ({"context": retriever, "question": RunnablePassthrough()}
         | prompt
         | llm
         | StrOutputParser()
         )

In [ ]:
print(chain.invoke("how can LLM hallucination impact a RAG application?"))

### Exercise

In [ ]:
# Understanding the BM25 retriever
chunks = [
    "RAG stands for Retrieval Augmented Generation.",
    "Graph Retrieval Augmented Generation uses graphs to store and utilize relationships between documents in the retrieval process.",
    "There are different types of RAG architectures; for example, Graph RAG."
]

# Initialize the BM25 retriever
bm25_retriever = BM25Retriever.from_texts(chunks, k=3)

# Invoke the retriever
results = bm25_retriever.invoke("Graph RAG")

# Extract the page content from the first result
print("Most Relevant Document:")
print(results[0].page_content)

In [ ]:
# Create a BM25 retriever from chunks
retriever = BM25Retriever.from_documents(
    documents=chunks,
    k = 5
)

# Create the LCEL retrieval chain
chain = ({"context": retriever, "question": RunnablePassthrough()}
         | prompt
         | llm
         | StrOutputParser()
)

print(chain.invoke("What are knowledge-intensive tasks?"))